In [1]:
from core.dataset import get_dataloaders
from src.models.efficientNet import get_efficient_net

from src.task_3_balancing.train_focal import run as run_focal
from src.task_3_balancing.train_adasyn import run as run_adasyn
from src.task_3_balancing.train_mixup import run as run_mixup

import torch
import torch.optim as optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

train_loader, val_loader = get_dataloaders(
    "../data/raw/train",
    "../data/raw/test"
)

def get_model():
    return get_efficient_net(num_classes=7).to(device)

def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=3e-4)

def get_scheduler(opt):
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)


# =============================
# RUN ALL 3 STRATEGIES
# =============================

def run_strategy(name, run_fn):
    print(f"\n🔥 Running {name}")
    model = get_model()
    optimizer = get_optimizer(model)
    scheduler = get_scheduler(optimizer)
    run_fn(model, train_loader, val_loader, optimizer, scheduler, device)

run_strategy("FOCAL", run_focal)
run_strategy("MIXUP", run_mixup)
run_strategy("ADASYN", run_adasyn)


🔥 Running FOCAL
Strategy: focal
Model: efficientnet
Device: mps
Start Memory: 431.77 MB

Epoch 1/10
Train Loss: 0.8569
Train Acc: 48.59%
Val Acc:   61.48%
Macro F1:  0.5151
✅ New best model saved

Epoch 2/10
Train Loss: 0.6299
Train Acc: 59.55%
Val Acc:   64.61%
Macro F1:  0.5720
✅ New best model saved

Epoch 3/10
Train Loss: 0.5564
Train Acc: 62.79%
Val Acc:   66.65%
Macro F1:  0.6169
✅ New best model saved

Epoch 4/10
Train Loss: 0.5042
Train Acc: 65.37%
Val Acc:   67.58%
Macro F1:  0.6378
✅ New best model saved

Epoch 5/10
Train Loss: 0.4477
Train Acc: 68.14%
Val Acc:   68.31%
Macro F1:  0.6609
✅ New best model saved

Epoch 6/10
Train Loss: 0.4021
Train Acc: 70.57%
Val Acc:   68.78%
Macro F1:  0.6692
✅ New best model saved

Epoch 7/10
Train Loss: 0.3632
Train Acc: 72.25%
Val Acc:   69.39%
Macro F1:  0.6773
✅ New best model saved

Epoch 8/10
Train Loss: 0.3245
Train Acc: 74.54%
Val Acc:   70.38%
Macro F1:  0.6928
✅ New best model saved

Epoch 9/10
Train Loss: 0.3002
Train Acc: 75.91

In [3]:
from core.metrics import evaluate_model, generate_comparison

class_names = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]

results = []

# Load best models
for strategy in ["focal", "adasyn", "mixup"]:
    model = get_model()
    model.load_state_dict(torch.load(f"../results/checkpoints/{strategy}_best.pth"))
    model.to(device)

    res = evaluate_model(
        model,
        val_loader,
        device,
        class_names,
        strategy_name=strategy
    )

    results.append(res)

generate_comparison(results)

✅ Comparison table saved to: /Users/bsama/Desktop/ds_interns_project_2_2026-Bhavya_Emotion_Recognition_from_Faces/emotion-recognition/results/comparison_table.md
